# Phase 5 — Monitoring System & Automation

**Goal:** Simulate the automated monitoring pipeline across multiple runs and scenarios, then visualize the trends that Power BI will display.

## Steps
1. Run all 4 scenarios once
2. Simulate 10 runs of gradual drift
3. Plot F1 trend over time
4. Plot drift ratio over time
5. Alert log review

In [1]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path

sns.set_style('whitegrid')
EXPORTS_DIR = Path('../data/monitoring_exports')

## 1. Run All Scenarios Once

In [2]:
from src.monitoring import run_monitoring_pipeline

scenarios = ['baseline', 'noise', 'speaker_split', 'gradual']
results = {}

for scenario in scenarios:
    print(f'\n--- Running scenario: {scenario} ---')
    alert = run_monitoring_pipeline(
        config_path='../configs/config.yaml',
        features_dir='../data/features',
        models_dir='../models',
        exports_dir='../data/monitoring_exports',
        scenario=scenario,
        batch_size=100,
    )
    results[scenario] = alert


--- Running scenario: baseline ---

Simulating incoming batch — scenario: 'baseline' | size: 100
  Inference done — Acc: 0.6200  F1: 0.5904

  MONITORING ALERT CHECK — 2026-06-06 20:20:41
  Scenario      : baseline
  KS Drift      : 2.9%  ✅ (threshold: 30%)
  F1 Score      : 0.5904  (baseline: 0.5986, drop: +0.0082)
  Perf Alert    : ✅ NO
  Overall       : ✅ OK
  Report saved → model_metrics.csv

--- Running scenario: noise ---


ParserError: Error tokenizing data. C error: Expected 10 fields in line 3, saw 13


## 2. Simulate 10 Runs of Gradual Drift
Mimics what happens when noise slowly increases over time (e.g. mic degrading)

In [ ]:
import time

print('Simulating gradual drift over 10 runs...')
for i, noise_level in enumerate(np.linspace(0.0, 2.0, 10)):
    print(f'  Run {i+1}/10  noise_std={noise_level:.2f}')
    run_monitoring_pipeline(
        config_path='../configs/config.yaml',
        features_dir='../data/features',
        models_dir='../models',
        exports_dir='../data/monitoring_exports',
        scenario='noise',
        batch_size=80,
        noise_std=noise_level if noise_level > 0 else 0.01,
    )
    time.sleep(0.5)  # small pause so timestamps differ

print('Done.')

## 3. Load Monitoring Log

In [ ]:
monitor_df = pd.read_csv(EXPORTS_DIR / 'monitoring_log.csv')
monitor_df['timestamp'] = pd.to_datetime(monitor_df['timestamp'])
monitor_df = monitor_df.sort_values('timestamp').reset_index(drop=True)

print(f'Total monitoring runs: {len(monitor_df)}')
monitor_df.tail(10)

## 4. F1 Trend Over Time

In [ ]:
alert_df = pd.read_csv(EXPORTS_DIR / 'alert_log.csv')
alert_df['timestamp'] = pd.to_datetime(alert_df['timestamp'])
alert_df = alert_df.sort_values('timestamp').reset_index(drop=True)

SCENARIO_COLORS = {
    'baseline':     '#27ae60',
    'noise':        '#e74c3c',
    'speaker_split':'#f39c12',
    'gradual':      '#9b59b6',
}

fig, axes = plt.subplots(2, 1, figsize=(13, 9), sharex=False)

# F1 over time
ax = axes[0]
for scenario, grp in alert_df.groupby('scenario'):
    ax.plot(range(len(grp)), grp['f1_score'], marker='o',
            label=scenario, color=SCENARIO_COLORS.get(scenario, '#3498db'), alpha=0.8)
ax.axhline(alert_df['baseline_f1'].iloc[0], color='gray',
           linestyle='--', linewidth=1.5, label='Baseline F1')
ax.set_title('F1 Score per Monitoring Run', fontsize=12)
ax.set_ylabel('F1-macro')
ax.set_ylim(0, 1.05)
ax.legend()

# Drift ratio over time
ax2 = axes[1]
for scenario, grp in alert_df.groupby('scenario'):
    ax2.plot(range(len(grp)), grp['ks_drift_ratio'], marker='s',
             label=scenario, color=SCENARIO_COLORS.get(scenario, '#3498db'), alpha=0.8)
ax2.axhline(0.3, color='red', linestyle='--', linewidth=1.5, label='Alert threshold (30%)')
ax2.set_title('KS Drift Ratio per Monitoring Run', fontsize=12)
ax2.set_ylabel('Drift Ratio')
ax2.set_ylim(0, 1.05)
ax2.legend()

plt.tight_layout()
plt.savefig(EXPORTS_DIR / 'monitoring_trends.png', dpi=120)
plt.show()

## 5. Alert Log Summary

In [ ]:
print('\nALERT LOG SUMMARY')
print('=' * 60)
summary = alert_df.groupby('scenario').agg(
    runs=('timestamp', 'count'),
    alerts=('any_alert', 'sum'),
    avg_f1=('f1_score', 'mean'),
    avg_drift=('ks_drift_ratio', 'mean'),
).round(4)
summary['alert_rate'] = (summary['alerts'] / summary['runs']).round(2)
print(summary.to_string())

# Save summary for Power BI
summary.to_csv(EXPORTS_DIR / 'monitoring_summary.csv')
print(f'\nSaved monitoring_summary.csv')

## 6. How to Use with n8n / Kestra

### n8n
```
1. Start n8n:  npx n8n
2. Open http://localhost:5678
3. Import pipeline/n8n_workflow.json
4. Update the project path in the Execute Command node
5. Add email credentials → Activate
```

### Kestra
```
1. Start Kestra:  docker-compose up (see kestra.io/docs)
2. Upload pipeline/kestra_flow.yaml via UI or API
3. Set ALERT_EMAIL secret in Kestra Settings
4. Trigger manually or wait for hourly cron
```

### Python Scheduler (lightweight alternative)
```bash
# Run every 60 seconds (for demo)
python src/monitoring.py --schedule 60 --scenario baseline

# Simulate noise drift every 30 seconds
python src/monitoring.py --schedule 30 --scenario noise --noise_std 1.5
```